# Basic 02 - Semantic Normalization
Small, linear flow for semantic mapping, strict controls, and validation diagnostics.


## 1) Imports


In [ ]:
from pathlib import Path
import sys
import warnings
import logging

import pandas as pd
from IPython.display import display


## 2) Quiet logs (optional)


In [ ]:
warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)


## 3) Make local package importable


In [ ]:
def _ensure_local_package() -> None:
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]
    for root in search_roots:
        if (root / "isa_phm").is_dir() and (root / "pyproject.toml").exists():
            root_s = str(root)
            if root_s not in sys.path:
                sys.path.insert(0, root_s)
            return
    raise RuntimeError("Could not locate python-wrapper root with isa_phm package.")

_ensure_local_package()


## 4) Import wrapper + errors


In [ ]:
from isa_phm import ISAWrapper
from isa_phm.errors import ValidationError


## 5) Pick ISA JSON


In [ ]:
ISA_JSON = Path(r"g:/ISA/ISA-PHM-Wizard/src/tests/fixtures/golden/isa-phm-out-milling.json")
ISA_JSON


## 6) Optional semantic override config path


In [ ]:
# Keep None if you don't use a project-specific override config.
SEMANTIC_OVERRIDE = None
# Example:
# SEMANTIC_OVERRIDE = Path(r"g:/ISA/my-semantic-overrides.json")
SEMANTIC_OVERRIDE


## 7) Build wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=ISA_JSON.parent,
    strict_validation=False,
    semantic_config_path=SEMANTIC_OVERRIDE,
)


## 8) Build semantic manifest


In [ ]:
manifest = wrapper.semantic_manifest()


## 9) Manifest diagnostics


In [ ]:
display(pd.DataFrame([manifest.diagnostics.model_dump()]))


## 10) Semantic factors for first study


In [ ]:
study = wrapper.study(wrapper.list_studies()[0].title)
sem_factors = study.semantic_factors()
sem_factors_df = pd.DataFrame([f.model_dump() for f in sem_factors])

display(sem_factors_df[["source_name", "semantic_key", "status", "confidence", "provenance"]])


## 11) Semantic parameters for first assay


In [ ]:
assay = study.assay(study.list_assays()[0].assay_id)
sem_params = assay.semantic_parameters()
meas_df = pd.DataFrame([p.model_dump() for p in sem_params["measurement"]])
proc_df = pd.DataFrame([p.model_dump() for p in sem_params["processing"]])


## 12) Measurement parameter mappings


In [ ]:
display(meas_df[["source_name", "semantic_key", "status", "confidence", "provenance"]])


## 13) Processing parameter mappings


In [ ]:
display(proc_df[["source_name", "semantic_key", "status", "confidence", "provenance"]])


## 14) Strict semantic check example


In [ ]:
try:
    strict_manifest = wrapper.semantic_manifest(
        strict=True,
        max_unknown_ratio=0.05,
        max_ambiguous_ratio=0.0,
        require_override_config=False,
    )
    print("Strict semantic validation passed.")
    display(pd.DataFrame([strict_manifest.diagnostics.model_dump()]))
except ValidationError as exc:
    print("Strict semantic validation failed:")
    print(exc)


## 15) Dataset validation report (semantic + metadata checks)


In [ ]:
report = wrapper.validate_dataset(
    check_files=True,
    semantic_strict=False,
    max_unknown_ratio=0.05,
    max_ambiguous_ratio=0.0,
    require_override_config=False,
)
summary = {
    "ok": report.ok,
    "n_errors": report.n_errors,
    "n_warnings": report.n_warnings,
    "n_info": report.n_info,
}
display(pd.DataFrame([summary]))


## 16) First validation issues


In [ ]:
issues_df = pd.DataFrame([i.model_dump() for i in report.issues])
display(issues_df.head(20))


## 17) AI context export with semantic + validation sections


In [ ]:
ai_ctx = wrapper.ai_context(include_semantics=True, include_validation=True)
list(ai_ctx.keys())


## 18) AI semantic diagnostics snippet


In [ ]:
display(pd.DataFrame([ai_ctx["semantic_manifest"]["diagnostics"]]))
